In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the count directly. Could you specify:
- Which database/system you’re using (PostgreSQL, MySQL, SQLite, MongoDB, etc.)
- The table or collection that stores artists (and whether you want distinct artists by an ID or by name)

In the meantime, here are common ways to get the count:

SQL databases (PostgreSQL/MySQL/SQLite)
- Total rows in the artists table:
  SELECT COUNT(*) AS artist_count FROM artists;

- Distinct artists by id (if there can be duplicates):
  SELECT COUNT(DISTINCT id) AS artist_count FROM artists;

- Distinct artists by name (if IDs aren’t reliable):
  SELECT COUNT(DISTINCT name) AS artist_count FROM artists;

MongoDB
- Count documents in the artists collection:
  db.artists.countDocuments({})

Python/pandas (if you’re loading data into a DataFrame)
- Total entries:
  len(artists_df)

- Distinct artists by id:
  artists_df['id'].nunique()

If you share your database type and the artists table definition (or paste a s